# 💾 Сохранение данных: как сделать объекты постоянными

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">От временных объектов к вечным данным</h2>
  <p>Вы уже умеете создавать классы и объекты. Но когда программа завершается, все данные исчезают. Как сделать так, чтобы объекты «жили» между запусками? Как сохранять и загружать состояние?</p>
  <p>В этой лекции вы научитесь:</p>
  <ul>
    <li>Работать с файлами: читать, писать, использовать контекстные менеджеры</li>
    <li>Сериализовать объекты в JSON и другие форматы</li>
    <li>Использовать <code>pickle</code> для сохранения любых объектов Python</li>
    <li>Проектировать классы с методами <code>save()</code> и <code>load()</code></li>
  </ul>
</div>

## 🎯 Цели лекции

- Понять разницу между временными и постоянными данными
- Научиться сохранять и загружать объекты в текстовые файлы (CSV, JSON)
- Освоить бинарную сериализацию с помощью `pickle`
- Узнать, какие форматы подходят для разных задач
- Реализовать в классах методы `save()`, `load()` и `from_file()`
- Изучить безопасность и ограничения разных подходов

## 📚 План лекции

1. **Почему данные исчезают?** — оперативная память vs постоянное хранилище
2. **Основы работы с файлами в Python**
   - Режимы открытия (`r`, `w`, `a`, `b`, `+`)
   - Контекстный менеджер `with`
   - Чтение и запись строк
3. **Текстовые форматы: CSV и JSON**
   - Модуль `csv` – для табличных данных
   - Модуль `json` – универсальный обмен данными
   - Преобразование объектов в словари и обратно
4. **Бинарная сериализация: `pickle`**
   - Как сохранить любой объект Python
   - Ограничения и безопасность
   - Альтернативы: `shelve`, `marshal`
5. **Проектирование классов с методами сохранения**
   - Паттерн `save()` / `load()`
   - Классовые методы `from_file()`, `from_json()`
   - Сохранение коллекций объектов
6. **Сравнение форматов** — когда что использовать
7. **Практические примеры**
   - Записная книжка (сохранение в JSON)
   - Библиотека книг (CSV + pickle)
8. **Итоги и рекомендации**

In [ ]:
# Проблема: данные живут только во время работы программы
class Note:
    def __init__(self, title, content):
        self.title = title
        self.content = content

notes = [Note("Первый", "Привет"), Note("Второй", "Мир")]
# после закрытия программы все заметки исчезнут

## 1. Почему данные исчезают?

Оперативная память (RAM) – быстрая, но **энергозависимая**. При выключении компьютера или завершении программы данные теряются.

Чтобы данные сохранялись между запусками, их нужно записать в **постоянное хранилище** – файловую систему (жёсткий диск, SSD). Python предоставляет инструменты для работы с файлами и сериализации – преобразования объектов в поток байтов (или текст) и обратно.

## 2. Основы работы с файлами в Python

### Открытие файла
```python
file = open("filename.txt", mode="r")
# работа с файлом
file.close()

In [ ]:
# Запись в файл
with open("example.txt", "w", encoding="utf-8") as f:
    f.write("Привет, мир!\n")
    f.write("Вторая строка")

# Чтение из файла
with open("example.txt", "r", encoding="utf-8") as f:
    content = f.read()
    print(content)

# Построчное чтение
with open("example.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

## 3. Текстовые форматы: CSV и JSON

### CSV (Comma-Separated Values) – для табличных данных

In [ ]:
import csv

# Запись списка объектов в CSV
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

students = [Student("Alice", 85), Student("Bob", 92)]

with open("students.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "score"])   # заголовок
    for s in students:
        writer.writerow([s.name, s.score])

# Чтение из CSV
loaded = []
with open("students.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)   # читает первую строку как заголовки
    for row in reader:
        loaded.append(Student(row["name"], int(row["score"])))

for s in loaded:
    print(s.name, s.score)

### JSON – универсальный формат обмена данными

JSON поддерживает: числа, строки, булевы значения, списки, словари. Встроенный модуль `json` легко преобразует эти типы.

In [ ]:
import json

class Book:
    def __init__(self, title, author, year):
        self.title = title
        self.author = author
        self.year = year

    def to_dict(self):
        """Преобразует объект в словарь для JSON."""
        return {"title": self.title, "author": self.author, "year": self.year}

    @classmethod
    def from_dict(cls, data):
        """Создаёт объект из словаря."""
        return cls(data["title"], data["author"], data["year"])

# Сохранение списка книг
books = [Book("1984", "Orwell", 1949), Book("Brave New World", "Huxley", 1932)]
with open("books.json", "w", encoding="utf-8") as f:
    json.dump([b.to_dict() for b in books], f, ensure_ascii=False, indent=2)

# Загрузка
with open("books.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    loaded_books = [Book.from_dict(item) for item in data]

for b in loaded_books:
    print(b.title, b.author, b.year)

## 4. Бинарная сериализация: pickle

`pickle` позволяет сохранять **любые** объекты Python – классы, функции, замыкания – в бинарном виде. Но **небезопасен** (не загружайте данные из ненадёжных источников) и не универсален для других языков.

In [ ]:
import pickle

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def greet(self):
        return f"Hi, I'm {self.name}"

# Сохранение
p = Person("Elena", 28)
with open("person.pkl", "wb") as f:
    pickle.dump(p, f)

# Загрузка
with open("person.pkl", "rb") as f:
    loaded_p = pickle.load(f)

print(loaded_p.name, loaded_p.age)
print(loaded_p.greet())

### Ограничения pickle:
- Не читается другими языками
- Небезопасен: злонамеренные данные могут выполнить код
- Может быть несовместим между версиями Python
- Не работает с некоторыми объектами (файлы, сокеты)

### Альтернативы:
- `shelve` – хранилище «ключ-значение» поверх pickle
- `marshal` – для внутреннего использования Python (не рекомендуется)
- JSON, YAML, XML – для обмена данными

## 5. Проектирование классов с методами сохранения

### Паттерн: `save()` и `load()` (или `from_file`)

In [ ]:
class Config:
    def __init__(self, setting1, setting2):
        self.setting1 = setting1
        self.setting2 = setting2

    def save(self, filename):
        with open(filename, "w") as f:
            json.dump(self.__dict__, f)

    @classmethod
    def load(cls, filename):
        with open(filename, "r") as f:
            data = json.load(f)
        return cls(**data)

# Использование
cfg = Config("value1", 42)
cfg.save("config.json")

cfg2 = Config.load("config.json")
print(cfg2.setting1, cfg2.setting2)

### Сохранение коллекции объектов (менеджер)

In [ ]:
class Library:
    def __init__(self, name):
        self.name = name
        self.books = []   # список объектов Book

    def add_book(self, book):
        self.books.append(book)

    def save(self, filename):
        data = {
            "name": self.name,
            "books": [b.to_dict() for b in self.books]
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, filename):
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
        lib = cls(data["name"])
        for book_data in data["books"]:
            lib.books.append(Book.from_dict(book_data))
        return lib

# Использование
lib = Library("MyLib")
lib.add_book(Book("Python Crash Course", "Eric Matthes", 2019))
lib.save("library.json")

lib2 = Library.load("library.json")
print(lib2.name, len(lib2.books))

## 6. Сравнение форматов

| Формат | Читаемость | Скорость | Поддерживаемые типы | Безопасность | Кроссплатформенность |
|--------|------------|----------|---------------------|--------------|----------------------|
| Текст (plain) | Высокая | Низкая | Только строки | Высокая | Да |
| CSV | Высокая | Средняя | Простые (числа, строки) | Высокая | Да |
| JSON | Высокая | Средняя | Словари, списки, примитивы | Высокая | Да |
| YAML | Высокая | Низкая | Расширенные | Средняя | Да |
| pickle | Нет (бинарный) | Высокая | Любые Python-объекты | Низкая | Нет |
| shelve | Нет | Средняя | Любые (как pickle) | Низкая | Нет |

### Рекомендации:
- **JSON** – лучший выбор для большинства приложений: прост, читаем, безопасен.
- **CSV** – если данные табличные и будут открываться в Excel.
- **pickle** – только для временных данных внутри одного проекта, когда важна скорость и не нужна совместимость.
- **Текстовые файлы** – для логов, конфигураций с простой структурой.

## 7. Практические примеры

### Пример 1. Записная книжка (сохранение в JSON)

In [ ]:
class Notebook:
    def __init__(self):
        self.notes = []

    def add_note(self, title, content):
        self.notes.append({"title": title, "content": content, "id": len(self.notes)+1})

    def save(self, filename):
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(self.notes, f, ensure_ascii=False, indent=2)

    def load(self, filename):
        with open(filename, "r", encoding="utf-8") as f:
            self.notes = json.load(f)

    def show(self):
        for n in self.notes:
            print(f"{n['id']}: {n['title']} - {n['content'][:30]}")

nb = Notebook()
nb.add_note("Купить", "Молоко, хлеб, яйца")
nb.add_note("Встреча", "Завтра в 15:00")
nb.save("notes.json")

nb2 = Notebook()
nb2.load("notes.json")
nb2.show()

### Пример 2. CSV + pickle: двойное сохранение (для отчётов и для быстрой загрузки)

In [ ]:
class Report:
    def __init__(self, data):
        self.data = data

    def to_csv(self, filename):
        import csv
        with open(filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["x", "y"])
            writer.writerows(self.data)

    def save_pickle(self, filename):
        with open(filename, "wb") as f:
            pickle.dump(self.data, f)

    @classmethod
    def load_pickle(cls, filename):
        with open(filename, "rb") as f:
            data = pickle.load(f)
        return cls(data)

# Использование
r = Report([(1,2), (3,4), (5,6)])
r.to_csv("report.csv")
r.save_pickle("report.pkl")

r2 = Report.load_pickle("report.pkl")
print(r2.data)

## 8. Итоги и ключевые моменты

- **Файлы** – основа долговременного хранения. Всегда используйте `with` для автоматического закрытия.
- **JSON** – лучший формат для обмена данными и хранения объектов (через преобразование в словари).
- **CSV** – удобен для таблиц и работы с электронными таблицами.
- **pickle** – мощный, но опасный. Используйте только для своих данных в доверенной среде.
- **Проектируйте классы** с методами `to_dict()` / `from_dict()` или `save()` / `load()` – это делает код модульным.

</details>

---

## 📖 Домашнее задание

1. **Сохраняемая записная книжка**  
   Расширьте класс `Notebook` из примера: добавьте удаление заметок, поиск по заголовку. Реализуйте автоматическое сохранение при каждом изменении (вызов `save` внутри методов). При загрузке программы, если файл существует – загружать его.

2. **Менеджер контактов**  
   Создайте класс `Contact` с полями `name`, `phone`, `email`. Класс `ContactBook` хранит список контактов. Реализуйте методы `add`, `remove`, `search`. Сохраняйте книгу в JSON. Добавьте возможность экспорта в CSV.

3. **Игра с сохранением состояния**  
   Реализуйте простую текстовую игру (например, угадай число) с классом `GameState`, который хранит историю попыток, текущий рекорд. Добавьте методы `save()` и `load()`, чтобы можно было продолжить игру после перезапуска.